# 3. Insider Risk, Communication Compliance, eDiscovery, and Audit

Not every risk comes from outside attackers. In this notebook we cover the Purview tools for monitoring *people* inside your organization, and for responding to legal or regulatory events:

1. **Insider Risk Management** - detect risky *behavior* (data theft, leaks).
2. **Communication Compliance** - detect risky *messages* (harassment, sensitive data in chat).
3. **eDiscovery** - find and preserve content for legal cases.
4. **Audit** - the activity log that underpins every investigation.

## Setup

All Purview features in this notebook are **simulated** in Python - no Microsoft 365 tenant required.

1. `cd security-certs/sc-900/04-compliance-and-purview && uv sync`
2. Pick the **`.venv` kernel** from the VS Code kernel picker (top-right).
3. If it's missing: `Cmd+Shift+P` -> *Developer: Reload Window*.

---
## Insider Risk Management

Insider risk management detects risky behavior by employees, contractors, and partners - both malicious *and* accidental.

### Common risk indicators

| Category | Example activities |
|----------|-------------------|
| **Data theft by departing users** | Mass downloads, USB copies, printing right before resignation |
| **Data leaks** | Sharing files with personal email, uploading to non-approved cloud storage |
| **Security policy violations** | Visiting malicious sites, disabling antivirus |
| **Patient data misuse** (healthcare) | Accessing records outside normal duties |
| **Priority-user violations** | Unusual activity by executives or privileged users |

### How it works

1. **Policies** define what to monitor and which users to prioritize.
2. **Signals** flow in from Microsoft 365, Defender for Endpoint, and HR connectors.
3. **Alerts** fire when activity patterns match risk indicators.
4. **Cases** are created for analysts to investigate the timeline.
5. **Actions** include escalation, referral to legal, or handing off to eDiscovery.

### Exam tips

- Insider risk is about **user behavior**, not external attacks.
- It uses **ML** to correlate signals *over time* (not just single events).
- Content is **pseudonymized** by default - analysts see "User A" until they escalate.

### Bad vs. Best: spotting a departing-employee data leak

- **BAD**: only look at today's events in isolation. 150 file downloads looks bad but not *obviously* malicious; a single rule can't tell.
- **BEST**: correlate HR signal (resignation) + download spikes + USB copies + after-hours access over a week. The *pattern* is what proves intent.

In [1]:
# BAD: single-event rule. One big download -> alert. Easy to bypass, lots of false positives.
def bad_detector(event):
    return 'ALERT' if 'Downloaded 150' in event else 'ok'

events = [
    'Downloaded 150 files from SharePoint',
    'Downloaded 149 files from SharePoint',
    'Copied 50 files to USB drive',
]
print('=== BAD: single-event detector ===')
for e in events:
    print(f'  {bad_detector(e):<6} {e}')
print('\nProblem: attacker downloads 149 files and bypasses the rule. No context (is user resigning?).')

=== BAD: single-event detector ===
  ALERT  Downloaded 150 files from SharePoint
  ok     Downloaded 149 files from SharePoint
  ok     Copied 50 files to USB drive

Problem: attacker downloads 149 files and bypasses the rule. No context (is user resigning?).


In [2]:
# BEST: correlated, time-windowed behavioral score.
from collections import defaultdict

USER_ACTIVITIES = [
    # Normal user
    {'user': 'User A', 'day': -5, 'activity': 'Downloaded 3 files from SharePoint', 'risk_score': 5},
    {'user': 'User A', 'day': -4, 'activity': 'Shared document with team channel', 'risk_score': 0},
    {'user': 'User A', 'day': -3, 'activity': 'Edited budget spreadsheet', 'risk_score': 0},
    # Departing user with suspicious activity
    {'user': 'User B', 'day': -7, 'activity': 'Submitted resignation (HR signal)', 'risk_score': 20},
    {'user': 'User B', 'day': -5, 'activity': 'Downloaded 149 files (below simple threshold!)', 'risk_score': 40},
    {'user': 'User B', 'day': -4, 'activity': 'Copied 50 files to USB drive', 'risk_score': 45},
    {'user': 'User B', 'day': -3, 'activity': 'Forwarded 30 emails to personal account', 'risk_score': 50},
    {'user': 'User B', 'day': -2, 'activity': 'Accessed files at 2 AM (outside normal pattern)', 'risk_score': 35},
    # Accidental leak
    {'user': 'User C', 'day': -3, 'activity': 'Uploaded confidential doc to personal Dropbox', 'risk_score': 60},
    {'user': 'User C', 'day': -1, 'activity': 'Shared file with external email (competitor domain)', 'risk_score': 70},
]

user_risk = defaultdict(lambda: {'activities': [], 'total_risk': 0})
for act in USER_ACTIVITIES:
    user_risk[act['user']]['activities'].append(act)
    user_risk[act['user']]['total_risk'] += act['risk_score']

RISK_THRESHOLDS = {'low': 30, 'medium': 80, 'high': 120}

print('=== BEST: Insider Risk dashboard (correlated, 7-day window) ===\n')
for user, data in sorted(user_risk.items(), key=lambda x: -x[1]['total_risk']):
    risk = data['total_risk']
    if risk >= RISK_THRESHOLDS['high']:
        level = 'HIGH'
    elif risk >= RISK_THRESHOLDS['medium']:
        level = 'MEDIUM'
    elif risk >= RISK_THRESHOLDS['low']:
        level = 'LOW'
    else:
        level = 'NONE'

    print(f'[{level:<6}] {user} - score {risk}')
    for act in data['activities']:
        print(f'   Day {act["day"]:+d}  {act["activity"]}')
    if level in ('HIGH', 'MEDIUM'):
        print('   -> ACTION: create investigation case; hand off to eDiscovery if needed.')
    print()

=== BEST: Insider Risk dashboard (correlated, 7-day window) ===

[HIGH  ] User B - score 190
   Day -7  Submitted resignation (HR signal)
   Day -5  Downloaded 149 files (below simple threshold!)
   Day -4  Copied 50 files to USB drive
   Day -3  Forwarded 30 emails to personal account
   Day -2  Accessed files at 2 AM (outside normal pattern)
   -> ACTION: create investigation case; hand off to eDiscovery if needed.

[HIGH  ] User C - score 130
   Day -3  Uploaded confidential doc to personal Dropbox
   Day -1  Shared file with external email (competitor domain)
   -> ACTION: create investigation case; hand off to eDiscovery if needed.

[NONE  ] User A - score 5
   Day -5  Downloaded 3 files from SharePoint
   Day -4  Shared document with team channel
   Day -3  Edited budget spreadsheet



---
## Communication Compliance

Insider risk watches *file behavior*. **Communication Compliance** watches *conversations* in Teams, Exchange, Viva Engage (Yammer), and (with connectors) third-party chat platforms.

### What it detects

| Scenario | Example |
|---|---|
| Harassment or threats | Bullying language in a Teams DM |
| Sensitive data in chat | Employee posts a credit card number in a channel |
| Conflict of interest | Trader discussing material non-public info |
| Regulated-industry violations | Financial rep making unsuitable promises to clients |

Policies are scoped to users or groups, messages are **pseudonymized by default**, and matches go into a reviewer queue (often staffed by HR or Legal, not IT).

### Exam tip

Don't confuse Communication Compliance with DLP:

- **DLP** blocks sharing of *sensitive data* regardless of intent.
- **Communication Compliance** reviews *messages* for policy/behavior violations (harassment, insider trading, etc.).

In [3]:
# Simulated communication compliance policy.
import re

POLICIES = [
    {'name': 'Harassment', 'keywords': ['idiot', 'shut up', 'hate you', 'stupid']},
    {'name': 'Sensitive data in chat', 'regexes': [r'\b\d{3}-\d{2}-\d{4}\b', r'\b4[0-9]{12}(?:[0-9]{3})?\b']},
    {'name': 'Insider trading hints', 'keywords': ['before earnings', 'not public yet', 'guaranteed return']},
]

MESSAGES = [
    {'from': 'alice', 'to': 'bob',   'channel': 'Teams DM',  'text': 'Can you review the deck before noon?'},
    {'from': 'carol', 'to': 'dave',  'channel': 'Teams DM',  'text': 'You are such an idiot, stop asking.'},
    {'from': 'eve',   'to': 'frank', 'channel': 'Teams chan','text': 'Here is my SSN for the form: 123-45-6789'},
    {'from': 'grace', 'to': 'heidi', 'channel': 'Outlook',   'text': 'Buy the stock before earnings tomorrow, not public yet.'},
    {'from': 'ivan',  'to': 'judy',  'channel': 'Teams chan','text': 'Lunch at 1 pm?'},
]

def evaluate_message(msg: dict) -> list:
    hits = []
    text_lower = msg['text'].lower()
    for p in POLICIES:
        if any(k in text_lower for k in p.get('keywords', [])):
            hits.append(p['name'])
        elif any(re.search(r, msg['text']) for r in p.get('regexes', [])):
            hits.append(p['name'])
    return hits

print('=== Communication Compliance review queue ===\n')
for m in MESSAGES:
    hits = evaluate_message(m)
    flag = '[FLAG]' if hits else '[ok  ]'
    # Pseudonymize sender until a reviewer escalates.
    display_from = 'User-' + str(hash(m['from']) % 1000).zfill(3) if hits else m['from']
    print(f'{flag} {m["channel"]:<11} {display_from:<10} -> {m["to"]:<6}: {m["text"]}')
    if hits:
        print(f'        policies matched: {hits}')
        print('        -> routed to HR/Legal reviewer; sender name hidden until escalation.')

=== Communication Compliance review queue ===

[ok  ] Teams DM    alice      -> bob   : Can you review the deck before noon?
[FLAG] Teams DM    User-352   -> dave  : You are such an idiot, stop asking.
        policies matched: ['Harassment']
        -> routed to HR/Legal reviewer; sender name hidden until escalation.
[FLAG] Teams chan  User-057   -> frank : Here is my SSN for the form: 123-45-6789
        policies matched: ['Sensitive data in chat']
        -> routed to HR/Legal reviewer; sender name hidden until escalation.
[FLAG] Outlook     User-887   -> heidi : Buy the stock before earnings tomorrow, not public yet.
        policies matched: ['Insider trading hints']
        -> routed to HR/Legal reviewer; sender name hidden until escalation.
[ok  ] Teams chan  ivan       -> judy  : Lunch at 1 pm?


---
## eDiscovery

**eDiscovery** (electronic discovery) is the process of finding, preserving, and analyzing electronic information for legal cases, investigations, or regulatory requests.

### Three tiers in Microsoft Purview

| Tier | What it includes | Typical license |
|------|-----------------|----------|
| **Content search** | Search across Exchange, SharePoint, OneDrive, Teams | E3 |
| **eDiscovery (Standard)** | Content search + cases, holds, export | E3 |
| **eDiscovery (Premium)** | Standard + custodian management, review sets, analytics, predictive coding | E5 |

### Workflow: EDRM in one picture

```
1. Identify  -> who are the custodians (people involved)?
2. Preserve  -> put legal holds on their mailboxes and sites
3. Collect   -> search for relevant content across M365
4. Process   -> de-duplicate, filter, OCR images
5. Review    -> attorneys review documents (Premium: AI-assisted)
6. Export    -> package for production to opposing counsel
```

### Legal holds

A legal hold **preserves** all content for a custodian, even if they delete it. The user does not know their content is on hold. Content is preserved in a hidden `Recoverable Items` folder.

### Exam tip

eDiscovery is about **legal and regulatory** investigations. Don't confuse it with insider risk (behavior monitoring) or DLP (prevention).

In [4]:
# Simulate a small eDiscovery case end-to-end.
CASE = {'name': 'Project Phoenix litigation', 'custodians': ['alice@contoso.com', 'bob@contoso.com']}

TENANT_CONTENT = [
    {'user': 'alice@contoso.com', 'workload': 'Exchange',   'id': 'm1', 'subject': 'RE: Project Phoenix timeline'},
    {'user': 'alice@contoso.com', 'workload': 'SharePoint', 'id': 's1', 'subject': 'phoenix-contract-v3.docx'},
    {'user': 'bob@contoso.com',   'workload': 'Teams',      'id': 't1', 'subject': 'Phoenix kickoff chat'},
    {'user': 'bob@contoso.com',   'workload': 'OneDrive',   'id': 'o1', 'subject': 'lunch-menu.pdf'},
    {'user': 'carol@contoso.com', 'workload': 'Exchange',   'id': 'm2', 'subject': 'HR Newsletter'},
]

print(f'=== eDiscovery case: {CASE["name"]} ===\n')
print('[1] Identify custodians:', CASE['custodians'], '\n')

print('[2] Preserve - place legal hold:')
for c in CASE['custodians']:
    print(f'    hold placed on mailbox and OneDrive for {c}')
print()

query = 'phoenix'
print(f'[3] Collect - search for "{query}" scoped to custodians:')
hits = [
    c for c in TENANT_CONTENT
    if c['user'] in CASE['custodians'] and query in c['subject'].lower()
]
for h in hits:
    print(f'    hit: [{h["workload"]:<10}] {h["id"]} - {h["subject"]}')
print(f'    -> {len(hits)} items collected.\n')

print('[4] Process - deduplicate + OCR (no duplicates in this demo).')
print('[5] Review - attorneys tag items as responsive/privileged.')
print('[6] Export - package into a production ZIP for opposing counsel.')

=== eDiscovery case: Project Phoenix litigation ===

[1] Identify custodians: ['alice@contoso.com', 'bob@contoso.com'] 

[2] Preserve - place legal hold:
    hold placed on mailbox and OneDrive for alice@contoso.com
    hold placed on mailbox and OneDrive for bob@contoso.com

[3] Collect - search for "phoenix" scoped to custodians:
    hit: [Exchange  ] m1 - RE: Project Phoenix timeline
    hit: [SharePoint] s1 - phoenix-contract-v3.docx
    hit: [Teams     ] t1 - Phoenix kickoff chat
    -> 3 items collected.

[4] Process - deduplicate + OCR (no duplicates in this demo).
[5] Review - attorneys tag items as responsive/privileged.
[6] Export - package into a production ZIP for opposing counsel.


---
## Audit

Microsoft Purview **Audit** records user and admin activities across Microsoft 365. Every other tool in this notebook relies on the audit log as a source of truth.

### Two tiers

| | Audit (Standard) | Audit (Premium) |
|-|------------------|-----------------|
| **Retention** | 180 days | 1 year (extendable to 10 years) |
| **Events** | Core activities | All Standard + high-value events (mail items accessed, mail items sent) |
| **Access** | Search UI | Search UI + API access |
| **License** | E3 | E5 |

### What gets audited

| Service | Example events |
|---------|----------------|
| Exchange | Email sent, mail read, mailbox login |
| SharePoint | File downloaded, shared, deleted |
| Entra ID | User created, password changed, MFA registered |
| Teams | Meeting joined, message sent, channel created |
| Admin | Policy changed, role assigned, eDiscovery search run |

In [5]:
AUDIT_LOGS = [
    {'time': '2026-04-16 09:00', 'user': 'alice@contoso.com', 'activity': 'UserLoggedIn',           'service': 'Entra ID',   'detail': 'MFA completed'},
    {'time': '2026-04-16 09:05', 'user': 'alice@contoso.com', 'activity': 'FileDownloaded',         'service': 'SharePoint', 'detail': 'budget-2026.xlsx'},
    {'time': '2026-04-16 09:10', 'user': 'alice@contoso.com', 'activity': 'FileSyncDownloadedFull', 'service': 'OneDrive',   'detail': '47 files synced'},
    {'time': '2026-04-16 09:15', 'user': 'admin@contoso.com', 'activity': 'Add member to role',     'service': 'Entra ID',   'detail': 'bob -> Global Admin'},
    {'time': '2026-04-16 09:20', 'user': 'bob@contoso.com',   'activity': 'Set-Mailbox',            'service': 'Exchange',   'detail': 'Forwarding rule added'},
    {'time': '2026-04-16 09:30', 'user': 'carol@contoso.com', 'activity': 'SearchStarted',          'service': 'eDiscovery', 'detail': 'Search: "project alpha"'},
    {'time': '2026-04-16 10:00', 'user': 'system',            'activity': 'DLPRuleMatch',           'service': 'DLP',        'detail': 'SSN detected in email to external'},
]

def audit_search(service: str = None, user: str = None) -> list:
    results = AUDIT_LOGS
    if service:
        results = [r for r in results if r['service'] == service]
    if user:
        results = [r for r in results if r['user'] == user]
    return results

def print_logs(logs: list) -> None:
    for log in logs:
        print(f'{log["time"]:<17} {log["user"]:<22} {log["service"]:<12} {log["activity"]:<24} {log["detail"]}')

print('=== Full audit log ===')
print_logs(AUDIT_LOGS)

print('\n=== Filter: everything admin@contoso.com did ===')
print_logs(audit_search(user='admin@contoso.com'))

print('\n=== Filter: all SharePoint events ===')
print_logs(audit_search(service='SharePoint'))

print('\n[tip] Premium audit retains logs up to 10 years and adds MailItemsAccessed / MailItemsSent.')

=== Full audit log ===
2026-04-16 09:00  alice@contoso.com      Entra ID     UserLoggedIn             MFA completed
2026-04-16 09:05  alice@contoso.com      SharePoint   FileDownloaded           budget-2026.xlsx
2026-04-16 09:10  alice@contoso.com      OneDrive     FileSyncDownloadedFull   47 files synced
2026-04-16 09:15  admin@contoso.com      Entra ID     Add member to role       bob -> Global Admin
2026-04-16 09:20  bob@contoso.com        Exchange     Set-Mailbox              Forwarding rule added
2026-04-16 09:30  carol@contoso.com      eDiscovery   SearchStarted            Search: "project alpha"
2026-04-16 10:00  system                 DLP          DLPRuleMatch             SSN detected in email to external

=== Filter: everything admin@contoso.com did ===
2026-04-16 09:15  admin@contoso.com      Entra ID     Add member to role       bob -> Global Admin

=== Filter: all SharePoint events ===
2026-04-16 09:05  alice@contoso.com      SharePoint   FileDownloaded           budget-202

---
## Putting it all together: a suspicious-insider investigation

One realistic scenario that uses *every* tool in this notebook:

```
1. Audit log           -> captures Bob mass-downloading files.
2. Insider Risk Mgmt   -> correlates with his resignation and raises a High alert.
3. Comm. Compliance    -> flags his Teams DM: "send me everything before I leave".
4. DLP                 -> blocks the subsequent external email of payroll data.
5. eDiscovery + Hold   -> Legal opens a case, puts a hold on Bob's mailbox/OneDrive.
6. Audit (Premium)     -> 10-year retention preserves the evidence trail.
```

No single tool catches everything. The exam (and real life) is about knowing which one to reach for.

---
## SC-900 Compliance Domain Cheat Sheet

| Concept | Key fact |
|---------|----------|
| **Service Trust Portal** | Microsoft's audit reports (SOC, ISO, etc.) |
| **Compliance Manager** | Your compliance score + improvement actions |
| **Priva** | Privacy risk management + subject rights requests |
| **Sensitive info types** | Pattern matching (SSN, credit cards) |
| **Trainable classifiers** | ML-based content classification |
| **Sensitivity labels** | Classify + encrypt + watermark documents |
| **DLP** | Block/warn when sharing sensitive data |
| **Retention policies** | Keep/delete by age at location level |
| **Retention labels** | Keep/delete individual items, can be records |
| **Insider risk management** | Detect risky user behavior (data theft, leaks) |
| **Communication Compliance** | Detect risky messages (harassment, sensitive-data leaks in chat) |
| **eDiscovery** | Find + preserve + review data for legal cases |
| **Legal hold** | Preserve all content even if user deletes it |
| **Audit (Standard)** | 180-day activity logs |
| **Audit (Premium)** | 1-10 year logs + high-value events (E5) |

---
## You've completed all SC-900 labs!

### Next steps

1. Take the [SC-900 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/security-compliance-and-identity-fundamentals/practice/assessment?assessment-type=practice&assessmentId=11&practice-assessment-type=certification)
2. Review any weak areas in the Microsoft Learn modules.
3. Schedule the exam when you're consistently scoring 80%+ on practice tests.

### What's next in this repo

- **AZ-500** (Azure Security Engineer Associate) - hands-on Azure security implementation.
- **SC-100** (Cybersecurity Architect Expert) - designing security architectures.
